# WMT-Human — Analysis

Models:
- Mistral
- LLaMA
- GPT-4o

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

In [ ]:
# Load preprocessed data (run preprocessing.ipynb first if needed)
df = pd.read_csv('../data/wmt-human_en_de_judged_cleaned.csv')

print(f"Loaded cleaned dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head()

In [ ]:
# Define column names and score range for analysis
human_cols = [col for col in df.columns if col.startswith('human_')]
llm_cols = [col for col in df.columns if col.endswith('as_a_judge')]
MIN, MAX = 0, 6

print(f"Human columns: {human_cols}")
print(f"LLM columns: {llm_cols}")
print(f"Score range: [{MIN}, {MAX}]")

## Diversity of opinion

- initialize evaluator instance:

In [ ]:
df.columns

In [ ]:
# Fix import path - add parent directory to Python path
import sys
import os
sys.path.append(os.path.join(os.getcwd(), '..', '..', '..'))

from core.llm_good_enough import LLMGoodEnough

good_enough = LLMGoodEnough(
    df=df,
    human_cols=human_cols,
    llm_cols=llm_cols,
    min_score=MIN,
    max_score=MAX,
    verbosity=0
)

# visualize LLM-as-a-judge good enough
good_enough.visualize_good_enough(llm_col='LLAMA_as_a_judge', y_lim=0.6)

### 1. Human-Human disagreement

In [ ]:
human_human_disagreements = good_enough.compute_human_disagreements()

print(human_human_disagreements.shape)
# mean and std (=2 decimal places)
print(f"Mean: {round(human_human_disagreements.mean(), 2)}")
print(f"Std: {round(human_human_disagreements.std(), 2)}")

### 2. LLM-Human disagreement

In [ ]:
# print shapes, mean and std (=2 decimal places)
print(f"LLAMA:")
llama_human_disagreements = good_enough.compute_llm_human_disagreements('LLAMA_as_a_judge')
print(f"   - Shape: {llama_human_disagreements.shape}")
print(f"   - Mean: {round(llama_human_disagreements.mean(), 3)}")
print(f"   - Std: {round(llama_human_disagreements.std(), 3)}")
print()

print(f"MISTRAL:")
mistral_human_disagreements =  good_enough.compute_llm_human_disagreements('MISTRAL_as_a_judge')
print(f"   - Shape: {mistral_human_disagreements.shape}")
print(f"   - Mean: {round(mistral_human_disagreements.mean(), 3)}")
print(f"   - Std: {round(mistral_human_disagreements.std(), 3)}")
print()

print(f"GPT-4o:")
gpt4o_human_disagreements = good_enough.compute_llm_human_disagreements('GPT_as_a_judge')
print(f"   - Shape: {gpt4o_human_disagreements.shape}")
print(f"   - Mean: {round(gpt4o_human_disagreements.mean(), 3)}")
print(f"   - Std: {round(gpt4o_human_disagreements.std(), 3)}")

### 3. Random-Human disagreement


**Note:** computed internally during instance initialization.

In [ ]:
import random

# set seed for reproducibility
random.seed(42)

MIN, MAX = 0, 6

# compute random judge
df['RANDOM_as_a_judge'] = np.random.randint(MIN, MAX + 1, size=len(df))

# compute human-random judge disagreement
human_random_disagreements = good_enough.compute_llm_human_disagreements('RANDOM_as_a_judge')

# print shapes, mean and std (=2 decimal places)
print(f"RANDOM:")
print(f"   - Shape: {human_random_disagreements.shape}")
print(f"   - Mean: {round(human_random_disagreements.mean(), 3)}")
print(f"   - Std: {round(human_random_disagreements.std(), 3)}")

## Results

**Hypotheses (MWU, one-sided `alternative="greater"`):**
- **H₀:** LLM–Human disagreement is not greater than Human–Human disagreement.
- **H₁:** LLM–Human disagreement is greater.

In [ ]:
from scipy.stats import mannwhitneyu

# compute mann-whitney u test and round to 4 decimal places
llama_human_pvalue = mannwhitneyu(
    llama_human_disagreements, human_human_disagreements, alternative='greater'
).pvalue
print(f"LLAMA vs. Human: {round(llama_human_pvalue, 4)}")

gpt4o_human_pvalue = mannwhitneyu(
    gpt4o_human_disagreements, human_human_disagreements, alternative='greater'
).pvalue

print(f"GPT-4o vs. Human: {round(gpt4o_human_pvalue, 4)}")

mistral_human_pvalue = mannwhitneyu(
    mistral_human_disagreements, human_human_disagreements, alternative='greater'
).pvalue

print(f"MISTRAL vs. Human: {round(mistral_human_pvalue, 4)}")

random_human_pvalue = mannwhitneyu(
    human_random_disagreements, human_human_disagreements, alternative='greater'
).pvalue

print(f"RANDOM vs. Human: {round(random_human_pvalue, 4)}")

In [ ]:
good_enough.plot_judges_grid(
    bins=np.arange(0, 8),
    bar_width=0.35,
    y_lim=0.6,
    save_path="../svg/wmt_barplot.svg"
)

## Robustness (Monte Carlo)

In [ ]:
# --- Robustness visualization for all models in one figure ---

models = ['GPT', 'LLAMA', 'MISTRAL']
llm_cols = [f"{model}_as_a_judge" for model in models]

print("→ Running robustness analysis for all models ...")
good_enough.plot_monte_carlo_robustness_multi(
    llm_cols=llm_cols,
    iterations=33_000,
    save_path="../svg/wmt_monte_carlo_robustness.svg"
)

In [ ]:
good_enough.plot_human_stability_analysis(
    convergence_threshold=0.000005,
    min_iterations=20_000,
    max_iterations=500_000,
    parallel=True,
    save_path="../svg/wmt_human_stability.svg"
)

In [ ]:
good_enough.plot_human_stability_analysis(
    percentages=[10, 20, 50, 100, 150, 200, 250, 300, 350, 400, 450, 500],
    convergence_threshold=0.00001,
    min_iterations=1_000,
    max_iterations=100_000,
    save_path="../svg/wmt_human_stability_100plus.svg"
)